In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns


In [17]:
df=pd.read_csv('dirty_cafe_sales.csv')
print('Shape:', df.shape)
data.head(10)

Shape: (10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


In [15]:
print(' Data Types ')
print(df.dtypes)
print()
print(' Missing Values (NaN)')
print(df.isnull().sum())

 Data Types 
Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object

 Missing Values (NaN)
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64


In [19]:
for col in ['Item', 'Payment Method', 'Location']:
    print(df[col].value_counts(dropna=False))
    print()

Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64

Payment Method
NaN               2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64

Location
NaN         3265
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64



In [20]:
df.replace(['UNKNOWN', 'ERROR'], np.nan, inplace=True)

print('Item unique values:', df['Item'].unique())
print('Payment Method unique values:', df['Payment Method'].unique())
print('Location unique values:', df['Location'].unique())

Item unique values: ['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' nan 'Sandwich' 'Juice' 'Tea']
Payment Method unique values: ['Credit Card' 'Cash' nan 'Digital Wallet']
Location unique values: ['Takeaway' 'In-store' nan]


In [27]:
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')

df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
print(df[['Total Spent', 'Transaction Date','Quantity','Price Per Unit']].dtypes)

Total Spent                float64
Transaction Date    datetime64[ns]
Quantity                   float64
Price Per Unit             float64
dtype: object


Recalculate `Total Spent` 

In [28]:
missing_before = df['Total Spent'].isnull().sum()
print(f'Missing Total Spent BEFORE fix: {missing_before}')


Missing Total Spent BEFORE fix: 502


In [66]:
mask = df['Total Spent'].isnull()
df.loc[mask, 'Total Spent'] = df.loc[mask, 'Quantity'] * df.loc[mask,'Price Per Unit']

missing_after = df['Total Spent'].isnull().sum()
print(f'Missing Total Spent AFTER fix:  {missing_after}')
print(f'Recovered {missing_before - missing_after} rows!')

Missing Total Spent AFTER fix:  31
Recovered 471 rows!


In [33]:
df.dropna(subset=['Transaction Date'], inplace=True)
print(f'Rows after dropping missing dates: {len(df)}')

Rows after dropping missing dates: 9540


In [44]:
df['Item'].isnull().sum()

927

In [42]:
df['Price Per Unit'].isnull().sum()

506

In [64]:
replacements = {
    2: 'Coffee',
    1.5: 'Tea',
    5: 'Salad',
    1: 'Cookie',
}

for price, item in replacements.items():
    
    mask_item = (df['Price Per Unit'] == price) & (df['Item'].isna())
    df.loc[mask_item, 'Item'] = item
    
  
    mask_price = (df['Item'] == item) & (df['Price Per Unit'].isna())
    df.loc[mask_price, 'Price Per Unit'] = price

In [65]:
df['Item'].isnull().sum()

456

In [56]:
df['Price Per Unit'].isnull().sum()

267

In [49]:
df['Quantity'].isnull().sum()

454

In [62]:
mask = df['Quantity'].isna()
df.loc[mask, 'Quantity'] = df.loc[mask, 'Total Spent']  / df.loc[mask, 'Price Per Unit']

In [63]:
df['Quantity'].isnull().sum()

29

In [60]:
mask = df['Price Per Unit'].isna()
df.loc[mask, 'Price Per Unit'] = df.loc[mask, 'Total Spent']  / df.loc[mask, 'Quantity']

In [61]:
df['Price Per Unit'].isnull().sum()

20

In [67]:
df.isnull().sum()

Transaction ID         0
Item                 456
Quantity              29
Price Per Unit        20
Total Spent           31
Payment Method      3015
Location            3779
Transaction Date       0
dtype: int64

In [68]:
# Drop rows where Transaction Date is missing (can't place them on a timeline)
df.dropna(subset=['Transaction Date'], inplace=True)
print(f'Rows after dropping missing dates: {len(df)}')

Rows after dropping missing dates: 9540


In [69]:
# Fill categorical columns with 'Unknown'
df['Item'].fillna('Unknown', inplace=True)
df['Payment Method'].fillna('Unknown', inplace=True)
df['Location'].fillna('Unknown', inplace=True)

/var/folders/kw/rxg77qcn20bdq7ws7w7q7rbw0000gn/T/ipykernel_57566/971387130.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Item'].fillna('Unknown', inplace=True)
/var/folders/kw/rxg77qcn20bdq7ws7w7q7rbw0000gn/T/ipykernel_57566/971387130.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behav

In [70]:
df['Quantity'].fillna(df['Quantity'].median(), inplace=True)
df['Price Per Unit'].fillna(df['Price Per Unit'].median(), inplace=True)
df['Total Spent'].fillna(df['Total Spent'].median(), inplace=True)

# Confirm no more missing values
print('Missing values after all fixes:')
print(df.isnull().sum())

Missing values after all fixes:
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64


/var/folders/kw/rxg77qcn20bdq7ws7w7q7rbw0000gn/T/ipykernel_57566/2964610806.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Quantity'].fillna(df['Quantity'].median(), inplace=True)
/var/folders/kw/rxg77qcn20bdq7ws7w7q7rbw0000gn/T/ipykernel_57566/2964610806.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting

Add Useful Columns for the Dashboard

- `Month` — to group sales by month
- `Day of Week` — to see which days are busiest
- `Month Name` — human-readable month label

In [71]:
# Extract time-based features from the date column
df['Year']        = df['Transaction Date'].dt.year
df['Month']       = df['Transaction Date'].dt.month
df['Month Name']  = df['Transaction Date'].dt.strftime('%B')   # e.g. 'January'
df['Day of Week'] = df['Transaction Date'].dt.strftime('%A')   # e.g. 'Monday'

df[['Transaction Date', 'Year', 'Month', 'Month Name', 'Day of Week']].head(5)

,Transaction Date,Year,Month,Month Name,Day of Week
0,2023-09-08,2023,9,September,Friday
1,2023-05-16,2023,5,May,Tuesday
2,2023-07-19,2023,7,July,Wednesday
3,2023-04-27,2023,4,April,Thursday
4,2023-06-11,2023,6,June,Sunday


In [72]:
print('Final Shape')
print(df.shape)

print()
print(' Data Types')
print(df.dtypes)

print()
print(' Missing Values ')
print(df.isnull().sum())

print()
print('Numerical Summary ')
df[['Quantity', 'Price Per Unit', 'Total Spent']].describe()

Final Shape
(9540, 12)

 Data Types
Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
Year                         int32
Month                        int32
Month Name                  object
Day of Week                 object
dtype: object

 Missing Values 
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Year                0
Month               0
Month Name          0
Day of Week         0
dtype: int64

Numerical Summary 


,Quantity,Price Per Unit,Total Spent
count,9540.000000,9540.000000,9540.000000
mean,3.021384,2.947694,8.919025
std,1.417902,1.278308,5.997568
min,1.000000,1.000000,1.000000
25%,2.000000,2.000000,4.000000
50%,3.000000,3.000000,8.000000
75%,4.000000,4.000000,12.000000
max,5.000000,5.000000,25.000000


In [73]:
df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Year,Month,Month Name,Day of Week
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,2023,9,September,Friday
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16,2023,5,May,Tuesday
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19,2023,7,July,Wednesday
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27,2023,4,April,Thursday
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11,2023,6,June,Sunday
5,TXN_2602893,Smoothie,5.0,4.0,20.0,Credit Card,Unknown,2023-03-31,2023,3,March,Friday
6,TXN_4433211,Unknown,3.0,3.0,9.0,Unknown,Takeaway,2023-10-06,2023,10,October,Friday
7,TXN_6699534,Sandwich,4.0,4.0,16.0,Cash,Unknown,2023-10-28,2023,10,October,Saturday
8,TXN_4717867,Unknown,5.0,3.0,15.0,Unknown,Takeaway,2023-07-28,2023,7,July,Friday
9,TXN_2064365,Sandwich,5.0,4.0,20.0,Unknown,In-store,2023-12-31,2023,12,December,Sunday


In [74]:
df.to_csv('clean_cafe_sales.csv', index=False)

print('✅ Done! Saved as clean_cafe_sales.csv')
print(f'   Total rows: {len(df)}')
print(f'   Total columns: {len(df.columns)}')

✅ Done! Saved as clean_cafe_sales.csv
   Total rows: 9540
   Total columns: 12
